In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [4]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))

t = np.where(np.all([dates > datetime(2025,8,29), dates < datetime(2025,10,8)], axis=0))[0]
files = np.array(files)[t]

len(files)

122

In [ ]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 480


for i in range(0,720):

    view_new = View(nx, ny, xc, yc, Rsun, 0, i, -i / 2, rsun_arc=0) ### North pole view

    grid = view_new.grid()
    mean, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        data = undistort(data, header, xd, yd)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = (view_new.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=0.1, stonyhurst=False))
        grid_, alpha = transform(grid)
        data = bilinear(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        mean += np.nan_to_num((data - mean) * n / coverage)

    mean[coverage == 0] = np.nan
    show_data(mean, view_new, label=r'$B_{los}$', to_file=f'temp1/{i:03d}.png')